# Video Game Sales & Industry Data

Group B - Xavier Horisberger, Ismael Trentin

## Dataset

The dataset covers a timespan of 44 years, precisely from 1980 to 2024. It contains all sales (both digital and physical) of a game title, distinguishing from platform to platform (console) and other additional informations like genre and sales. In total, a sale is described by 14 columns. On the 64'016 rows of this dataset, we performed a variety of analysis using python and charting with plotly.

## Code

In [1]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

We import the dataset. To download it, run the shell script located in `data/download_dataset.sh`.

In [2]:
df = pd.read_csv("data/Video Games Sales (1980-2024) - Raw.csv")
df_cleaned = pd.read_csv("data/Video_Games_Sales_Cleaned.csv")

This is what we will work on:

In [3]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 64016 entries, 0 to 64015
Data columns (total 14 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   img           64016 non-null  str    
 1   title         64016 non-null  str    
 2   console       64016 non-null  str    
 3   genre         64016 non-null  str    
 4   publisher     64016 non-null  str    
 5   developer     63999 non-null  str    
 6   critic_score  6678 non-null   float64
 7   total_sales   18922 non-null  float64
 8   na_sales      12637 non-null  float64
 9   jp_sales      6726 non-null   float64
 10  pal_sales     12824 non-null  float64
 11  other_sales   15128 non-null  float64
 12  release_date  56965 non-null  str    
 13  last_update   17879 non-null  str    
dtypes: float64(6), str(8)
memory usage: 6.8 MB


These are the available consoles:

In [4]:
df["console"].unique()

<StringArray>
[   'PS3',    'PS4',    'PS2',   'X360',   'XOne',     'PC',    'PSP',
    'Wii',     'PS',     'DS',   '2600',    'GBA',    'NES',     'XB',
    'PSN',    'GEN',    'PSV',     'DC',    'N64',    'SAT',   'SNES',
    'GBC',     'GC',     'NS',    '3DS',     'GB',   'WiiU',     'WS',
     'VC',     'NG',     'WW',    'SCD',    'PCE',    'XBL',    '3DO',
     'GG',    'OSX',    'Mob',   'PCFX', 'Series',    'All',    'iOS',
   '5200',    'And',   'DSiW',   'Lynx',  'Linux',     'MS',    'ZXS',
   'ACPC',   'Amig',   '7800',    'DSi',     'AJ',   'WinP',   'iQue',
    'GIZ',     'VB',   'Ouya',  'NGage',    'AST',    'MSD',   'S32X',
     'XS',    'PS5',    'Int',     'CV',    'Arc',    'C64',    'FDS',
    'MSX',     'OR',   'C128',    'CDi',   'CD32',    'BRW',    'FMT',
   'ApII',    'Aco',   'BBCM',   'TG16']
Length: 81, dtype: str

Since the sales of a game title are distinguished by platform, the actual total number of game titles is:

In [5]:
len(df["title"].unique())

39798

To analyze game titles and not global sales, we aggregate the total sales by title:

In [6]:
sales_cols = ["total_sales", "na_sales", "jp_sales", "pal_sales", "other_sales"]

df_sales = df.copy()
# remove all NAs for thoes columns
df_sales[sales_cols] = df_sales[sales_cols].fillna(0)

df_sales["release_date"] = pd.to_datetime(
    df_sales["release_date"],
    format="%d-%m-%Y",
    errors="coerce"
)

df_agg_by_title = (
    df_sales.groupby("title", as_index=False)
    .agg({
        "genre": "first",
        "publisher": "first",
        "developer": "first",
        "critic_score": "mean",
        "release_date": "min",
        "total_sales": "sum",
        "na_sales": "sum",
        "jp_sales": "sum",
        "pal_sales": "sum",
        "other_sales": "sum"
    })
)

#df_agg_by_title.sort_values(by="total_sales", ascending=False)

In [7]:
fig = px.histogram(
    df_agg_by_title,
    x="total_sales",
    log_y=True,
    nbins=65,
    title="Distribution of Global Sales",
    labels={
        "total_sales": "Total sales [millions of copies]",
    }
)

fig.update_layout(yaxis_title="Number of games")
fig.update_traces(xbins_start=0)
fig.show()

We can observe that the top 5 game titles with the most sold copies, are all triple A games.

In [8]:
df_agg_by_title[df_agg_by_title['total_sales'] >= 28]

,title,genre,publisher,developer,critic_score,release_date,total_sales,na_sales,jp_sales,pal_sales,other_sales
5266,Call of Duty: Black Ops,Shooter,Activision,Treyarch,8.287500,2010-11-09,30.99,17.65,0.59,9.45,3.31
5273,Call of Duty: Black Ops II,Shooter,Activision,Treyarch,8.375000,2012-11-13,29.59,14.12,0.72,11.08,3.67
5277,Call of Duty: Ghosts,Shooter,Activision,Infinity Ward,7.833333,2013-11-05,28.80,15.06,0.49,9.60,3.65
5281,Call of Duty: Modern Warfare 3,Shooter,Activision,Infinity Ward,7.500000,2011-11-08,30.71,15.57,0.62,11.26,3.26
13724,Grand Theft Auto V,Action,Rockstar Games,Rockstar North,9.366667,2013-09-17,64.29,26.19,1.66,28.14,8.32


In [9]:
df_genres = df["genre"].value_counts().reset_index()

fig = px.bar(
    df_genres,
    x="count",
    y="genre",
    title="Ranking Genres by Number of Games",
    labels={
        "genre": "Genre",
        "count": "Number of games"
    }
)

fig.update_layout(
    yaxis = {'categoryorder': 'total ascending'},
    height = 600
)
fig.show()

In [10]:
# pandas qcut divide i dati in `q` gruppi con lo stesso numero di samples/osservazioni.
df_bplot = df[["critic_score", "total_sales", "title"]].dropna()
df_bplot["score_decile"] = pd.qcut(
    df_bplot["critic_score"],
    q=10,
    labels=[f"D{i}" for i in range(1, 11)]
)

fig = px.box(
    df_bplot,
    x="total_sales",
    y="score_decile",
    hover_name="title",
    hover_data=["critic_score"],
    log_x=True,  # consigliato perché le vendite sono molto skewed
    title="Total Sales Distribution by Critic Score Deciles",
    labels={
        "total_sales": "Total Sales (millions)",
        "score_decile": "Critic Score Decile"
    }
)

fig.update_layout(
    yaxis=dict(categoryorder="array",
               categoryarray=[f"D{i}" for i in range(1,11)])
)
fig.show()

In [11]:

sales_cols = ["na_sales", "pal_sales", "jp_sales", "other_sales"]

genre_region = (
    df.groupby("genre", as_index=False)[sales_cols]
    .sum()
)

genre_region["total_sales"] = genre_region[sales_cols].sum(axis=1)

genre_region = genre_region.sort_values("total_sales", ascending=True)

genre_region["North America"] = genre_region["na_sales"] / genre_region["total_sales"] * 100
genre_region["Europe (PAL)"] = genre_region["pal_sales"] / genre_region["total_sales"] * 100
genre_region["Japan"] = genre_region["jp_sales"] / genre_region["total_sales"] * 100
genre_region["Other"] = genre_region["other_sales"] / genre_region["total_sales"] * 100

fig = make_subplots(
    rows=1,
    cols=2,
    shared_yaxes=True,
    column_widths=[0.35, 0.65],
    horizontal_spacing=0.03,
    subplot_titles=[
        "Total sales by genre",
        "Regional sales share by genre"
    ]
)

fig.add_trace(
    go.Bar(
        x=genre_region["total_sales"],
        y=genre_region["genre"],
        orientation="h",
        name="Total sales",
        showlegend=False
    ),
    row=1,
    col=1
)

regions = ["North America", "Europe (PAL)", "Japan", "Other"]

for region in regions:
    fig.add_trace(
        go.Bar(
            x=genre_region[region],
            y=genre_region["genre"],
            orientation="h",
            name=region
        ),
        row=1,
        col=2
    )

fig.update_layout(
    title="Genre Preferences by Region",
    barmode="stack",
    xaxis_title="Total sales [millions of copies]",
    xaxis2_title="Regional share [%]",
    yaxis_title="Genre",
    height=650,
    legend_title="Region"
)

fig.update_xaxes(type="log", row=1, col=1)
fig.update_xaxes(range=[0, 100], row=1, col=2)

fig.show()

In [12]:
# we extract the year from the parsed released_date
df_time = df.copy()
df_time["release_date"] = pd.to_datetime(df_time["release_date"], format="%d-%m-%Y", errors="coerce").dropna()
df_time["year"] = df_time["release_date"].dt.year.dropna()

games_by_year = (
    df_time
    .groupby("year", as_index=False)["title"]
    .count()
    .rename(columns={"title": "games_count"})
)

fig = px.bar(
    games_by_year,
    x="year",
    y="games_count",
    title="Number of Released Games by Year",
    labels={
        "year": "Year",
        "games_count": "Number of games"
    }
)

fig.update_xaxes(range=[1978, games_by_year["year"].max()])

fig.show()

In [13]:
df["year"] = pd.to_datetime(df["release_date"], format="%d-%m-%Y").dt.year

df["console"] = df["console"].astype(str)
sales_by_console_year = (
    df.groupby(["year", "console"], as_index=False)["total_sales"]
    .sum()
)

sales_by_console_year = sales_by_console_year[sales_by_console_year["total_sales"] > 0]

# console più venduta per anno
top_console_per_year = (
    sales_by_console_year
    .sort_values(["year", "total_sales"], ascending=[True, False])
    .groupby("year")
    .head(1)
)

fig = px.bar(
    top_console_per_year,
    x="year",
    y="total_sales",
    color="console",
    title="Most popular console per year",
)
fig.update_xaxes(range=[1978, games_by_year["year"].max()])
fig.show()

Since we have a lot of records of game titles that do not register any sales, we have no bars starting from 2019.

In [14]:
top_consoles = (
    df.groupby("console")["total_sales"]
    .sum()
    .nlargest(10)
    .index
)

df_top = df[df["console"].isin(top_consoles)]

pivot = df_top.pivot_table(
    index="console",
    columns="genre",
    values="total_sales",
    aggfunc="sum"
)

fig = px.imshow(
    pivot,
    aspect="auto",
    color_continuous_scale="Viridis",
    title="Sales by genre for top consoles"
)
fig.update_layout(height=700)
fig.show()